# NVIDIA FastConformer-Hybrid (CTC) — LoRA fine-tuning (Arabic)

Sibling of `asr_qwen3_finetune.ipynb` (Qwen3-ASR) and `asr_model_agnostic_finetune.ipynb`
(OmniASR): **same pipeline shape** (ConfigAPI → ModelAdapter → PredictAPI → EvaluateAPI →
TrainAPI) but for **`nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0`** driven in **pure
CTC mode**.

### Why this model
The literal `nvidia/stt_ar_conformer_ctc_large` is **not downloadable** (404 on NGC and HF;
absent from NeMo's registry — see `models.md`). The official, ungated NVIDIA Arabic large
model is this **FastConformer Hybrid (Transducer+CTC)**, which carries a CTC head we drive
directly via `change_decoding_strategy(decoder_type="ctc")`. `pcd` = punctuation + capitalization
+ diacritics.

### Key differences from the Qwen3-ASR notebook
1. **CTC loss, not seq2seq.** Training is
   `enc,enc_len = model(input_signal, input_signal_length)` →
   `log_probs = model.ctc_decoder(encoder_output=enc)` →
   `loss = model.ctc_loss(log_probs, targets, input_lengths=enc_len, target_lengths=...)`.
   Targets are BPE ids from `model.tokenizer.text_to_ids(text)` (vocab 1024 + blank).
2. **NeMo, not transformers.** Loaded via `nemo_asr.models.ASRModel.from_pretrained` →
   `EncDecHybridRNNTCTCBPEModel` (114.6M). Inference is `model.transcribe([wav_paths])`
   → `[Hypothesis.text]` (CTC greedy).
3. **Real PEFT LoRA, in place.** The conformer attention/FF projections are `torch.nn.Linear`
   (`linear_q/k/v/out`, `linear1/2`), so `get_peft_model` injects `lora.Linear` **in place** —
   `self.model` stays the NeMo object so `.forward` / `.ctc_*` / `.transcribe` keep working;
   `self.peft` is the save handle.
4. **Separate venv.** NeMo 2.7.3 (`/workspace/venv_nemo_gpu`, torch 2.8 cu128) — conflicts
   with both the OmniASR (fairseq2) and Qwen3 (transformers 5.14) envs.


## Cell 1 — Environment

In [1]:
# Env is prebuilt at /workspace/venv_nemo_gpu (NeMo 2.7.3 + peft + torch cu128 + numpy<2 + mlflow + pandas).
# To rebuild elsewhere:
# !pip install "nemo_toolkit[asr]" "numpy<2.0" peft jiwer soundfile librosa bitsandbytes mlflow pandas
import os, json, gc, math, time, random, hashlib, shutil, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Callable

import numpy as np, torch, torch.nn as nn
warnings.filterwarnings("ignore")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
import logging; logging.getLogger("nemo_logger").setLevel(logging.ERROR)
print(torch.__version__, torch.cuda.is_available())

2.8.0+cu128 True


## Cell 2 — Config knobs

In [2]:
MODEL_NAME = "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0"

LANG          = "ar"
SMOKE_TEST    = True                # tiny subsets + 2 epochs
SEED          = 42

DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

ROOT          = Path(os.environ.get("ASR_ENV_ROOT", "/workspace/asr_env"))
MODEL_CACHE   = ROOT / "models"
PRED_DIR      = ROOT / "preds"
METRIC_DIR    = ROOT / "metrics"
CKPT_DIR      = ROOT / "checkpoints"
for d in (MODEL_CACHE, PRED_DIR, METRIC_DIR, CKPT_DIR): d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE / "hf")
import mlflow
# mlflow 3.x deprecated the plain "file:" tracking backend (raises unless
# MLFLOW_ALLOW_FILE_STORE=true) -- sqlite is the currently-recommended local backend and
# also lets `mlflow ui --backend-store-uri ...` browse runs later. Artifact location is
# set explicitly on first creation since sqlite backends don't default one on their own.
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{ROOT / 'mlflow.db'}")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
_MLFLOW_EXPERIMENT = "arabic-asr-conformer-ctc"
if mlflow.get_experiment_by_name(_MLFLOW_EXPERIMENT) is None:
    mlflow.create_experiment(_MLFLOW_EXPERIMENT, artifact_location=f"file:{ROOT / 'mlruns' / _MLFLOW_EXPERIMENT}")
mlflow.set_experiment(_MLFLOW_EXPERIMENT)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
print(f"DEVICE={DEVICE} | dtype={COMPUTE_DTYPE} | ROOT={ROOT}")

DEVICE=cuda | dtype=torch.bfloat16 | ROOT=/workspace/asr_env


## Cell 3 — Arabic normalization + WER/CER (identical to the Qwen3/OmniASR notebooks)

In [3]:
import re, unicodedata, jiwer

_DIAC = re.compile(r"[ؗ-ًؚ-ْـ]")
_PUNC = re.compile(r"[^\w\sء-ي]")

def normalize_ar(t: str) -> str:
    '''Diacritic strip, tatweel removal, alef/ya/ta-marbuta unification.'''
    if t is None: return ""
    t = unicodedata.normalize("NFKC", str(t))
    t = _DIAC.sub("", t)
    t = re.sub("[آأإٱ]", "ا", t)   # alef variants -> alef
    t = t.replace("ى", "ي")                          # alef maqsura -> ya
    t = t.replace("ة", "ه")                          # ta marbuta -> ha
    t = t.replace("ؤ", "و").replace("ئ", "ي")
    t = _PUNC.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()

def compute_wer_cer(preds, refs, normalize=True):
    if normalize:
        preds = [normalize_ar(p) for p in preds]
        refs  = [normalize_ar(r) for r in refs]
    keep = [(p, r) for p, r in zip(preds, refs) if r.strip()]
    if not keep: return {"wer": float("nan"), "cer": float("nan"), "n": 0}
    p, r = zip(*keep)
    return {"wer": jiwer.wer(list(r), list(p)),
            "cer": jiwer.cer(list(r), list(p)),
            "n": len(r)}

## Cell 4 — ConfigAPI

LoRA targets are the FastConformer encoder projections (all real `torch.nn.Linear`):
attention `linear_q/k/v/out` + feed-forward `linear1/2` across the 17 conformer layers.
`max_audio_seconds=30` keeps the smoke subset short; the encoder itself handles far longer.

In [4]:
@dataclass
class LoRAConfigSpec:
    r: int = 32
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    bias: str = "none"
    # FastConformer encoder attention + feed-forward projections (torch.nn.Linear).
    target_modules: List[str] = field(default_factory=lambda: [
        "linear_q","linear_k","linear_v","linear_out","linear1","linear2"])
    modules_to_save: Optional[List[str]] = None
    task_type: Optional[str] = None      # non-HF NeMo module -> leave None (LoRA still injects)

@dataclass
class TrainConfigSpec:
    num_epochs: int = 50
    early_stopping_patience: int = 3
    metric_for_best: str = "wer"
    greater_is_better: bool = False
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float = 1e-4
    warmup_ratio: float = 0.05
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    optim: str = "adamw_bnb_8bit"
    bf16: bool = True
    gradient_checkpointing: bool = False
    dataloader_num_workers: int = 0        # 0 avoids fork+CUDA issues (collate touches the tokenizer)
    max_audio_seconds: float = 30.0
    max_label_tokens: int = 256
    save_total_limit: int = 3
    save_steps: Optional[int] = None      # None -> auto: max(200, steps_per_epoch // 3)
    dataloader_pin_memory: bool = True
    dataloader_persistent_workers: bool = True
    dataloader_prefetch_factor: int = 4

class ConfigAPI:
    _LORA = {MODEL_NAME: LoRAConfigSpec()}
    _TRAIN = {MODEL_NAME: TrainConfigSpec()}
    @classmethod
    def lora(cls, name)  -> LoRAConfigSpec:  return cls._LORA.get(name, LoRAConfigSpec())
    @classmethod
    def train(cls, name) -> TrainConfigSpec: return cls._TRAIN.get(name, TrainConfigSpec())

print(json.dumps(asdict(ConfigAPI.lora(MODEL_NAME)), indent=2))
print(json.dumps(asdict(ConfigAPI.train(MODEL_NAME)), indent=2))

{
  "r": 32,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "bias": "none",
  "target_modules": [
    "linear_q",
    "linear_k",
    "linear_v",
    "linear_out",
    "linear1",
    "linear2"
  ],
  "modules_to_save": null,
  "task_type": null
}
{
  "num_epochs": 50,
  "early_stopping_patience": 3,
  "metric_for_best": "wer",
  "greater_is_better": false,
  "per_device_train_batch_size": 2,
  "per_device_eval_batch_size": 2,
  "gradient_accumulation_steps": 4,
  "learning_rate": 0.0001,
  "warmup_ratio": 0.05,
  "weight_decay": 0.0,
  "max_grad_norm": 1.0,
  "optim": "adamw_bnb_8bit",
  "bf16": true,
  "gradient_checkpointing": false,
  "dataloader_num_workers": 0,
  "max_audio_seconds": 30.0,
  "max_label_tokens": 256,
  "save_total_limit": 3,
  "save_steps": null,
  "dataloader_pin_memory": true,
  "dataloader_persistent_workers": true,
  "dataloader_prefetch_factor": 4
}


## Cell 5 — ModelAdapter ABC

In [5]:
from abc import ABC, abstractmethod

class ModelAdapter(ABC):
    name: str
    loss_type: str = "ctc"
    supports_unsloth: bool = False

    def __init__(self, model_name: str, lang: str = LANG):
        self.model_name = model_name; self.lang = lang
        self.model = None; self.peft = None

    @abstractmethod
    def load_base(self): ...
    @abstractmethod
    def preprocess(self, example: Dict) -> Dict: ...
    @abstractmethod
    def collate(self, features: List[Dict]) -> Dict[str, Any]: ...
    @abstractmethod
    def train_step(self, batch: Dict) -> "torch.Tensor": ...
    @abstractmethod
    def generate(self, batch: Dict) -> List[str]: ...

    # --- LoRA lifecycle (PEFT). CTC-model specific: inject in place, keep NeMo API on self.model.
    def apply_lora(self, spec: "LoRAConfigSpec"):
        from peft import LoraConfig, get_peft_model
        kw = dict(r=spec.r, lora_alpha=spec.lora_alpha, lora_dropout=spec.lora_dropout,
                  bias=spec.bias, target_modules=spec.target_modules)
        if spec.modules_to_save: kw["modules_to_save"] = spec.modules_to_save
        if spec.task_type:       kw["task_type"] = spec.task_type
        # get_peft_model injects lora.Linear into self.model's submodules IN PLACE and returns a
        # PeftModel wrapper. We keep self.model (the NeMo object) for forward/ctc/transcribe and
        # use self.peft only to save/print/reload the adapter.
        self.peft = get_peft_model(self.model, LoraConfig(**kw))
        self.peft.print_trainable_parameters()
        print("[lora] peft"); return self.peft

    def trainable_parameters(self):
        return [p for p in self.model.parameters() if p.requires_grad]

    def save_checkpoint(self, d):
        self.peft.save_pretrained(str(d))

    def load_checkpoint(self, d):
        from safetensors.torch import load_file
        from peft import set_peft_model_state_dict
        sd = load_file(str(Path(d) / "adapter_model.safetensors"))
        set_peft_model_state_dict(self.peft, sd)

## Cell 6 — FastConformer-CTC adapter

In [6]:
class ConformerCTCAdapter(ModelAdapter):
    '''nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0 in pure-CTC mode.
       Verified on GPU (2026-07-18):
         * load:  ASRModel.from_pretrained -> EncDecHybridRNNTCTCBPEModel (114.6M),
                  change_decoding_strategy(decoder_type="ctc")
         * TRAIN: enc,enc_len = model(input_signal, input_signal_length)
                  log_probs   = model.ctc_decoder(encoder_output=enc)
                  loss        = model.ctc_loss(log_probs, targets, enc_len, target_lengths)
         * INFER: model.transcribe([wav_paths]) -> [Hypothesis.text]   (CTC greedy)
         * LoRA:  real PEFT on conformer attention+FF linears, injected in place.
    '''
    loss_type = "ctc"
    supports_unsloth = False

    def load_base(self):
        import nemo.collections.asr as nemo_asr
        print(f"[load] {self.model_name}")
        self.model = nemo_asr.models.ASRModel.from_pretrained(self.model_name, map_location=DEVICE)
        self.model.change_decoding_strategy(decoder_type="ctc")   # drive the CTC head
        self.peft = None
        return self.model

    def preprocess(self, ex):
        audio = ex["audio"]["array"]; sr = ex["audio"]["sampling_rate"]
        if sr != 16000:
            import librosa
            audio = librosa.resample(np.asarray(audio, dtype=np.float32), orig_sr=sr, target_sr=16000)
        text = normalize_ar(ex["text"])
        return {"audio": np.asarray(audio, dtype=np.float32), "text": text,
                "audio_len": len(audio) / 16000.0}

    def collate(self, feats):
        sigs = [torch.from_numpy(np.asarray(f["audio"], dtype=np.float32)) for f in feats]
        lens = torch.tensor([int(s.numel()) for s in sigs], dtype=torch.long)
        maxT = int(lens.max())
        sig  = torch.zeros(len(sigs), maxT, dtype=torch.float32)
        for i, s in enumerate(sigs): sig[i, :s.numel()] = s
        toks = [self.model.tokenizer.text_to_ids(f["text"]) for f in feats]
        tlen = torch.tensor([len(t) for t in toks], dtype=torch.long)
        maxL = max(1, int(tlen.max()))
        tgt  = torch.zeros(len(toks), maxL, dtype=torch.long)
        for i, t in enumerate(toks):
            if t: tgt[i, :len(t)] = torch.tensor(t, dtype=torch.long)
        return {"input_signal": sig, "input_signal_length": lens,
                "targets": tgt, "target_lengths": tlen,
                "text": [f["text"] for f in feats], "_audio": [f["audio"] for f in feats]}

    def train_step(self, batch) -> torch.Tensor:
        enc, enc_len = self.model(input_signal=batch["input_signal"].to(DEVICE),
                                  input_signal_length=batch["input_signal_length"].to(DEVICE))
        log_probs = self.model.ctc_decoder(encoder_output=enc)
        return self.model.ctc_loss(log_probs=log_probs,
                                   targets=batch["targets"].to(DEVICE),
                                   input_lengths=enc_len,
                                   target_lengths=batch["target_lengths"].to(DEVICE))

    @torch.no_grad()
    def generate(self, batch):
        import tempfile, soundfile as sf
        was_training = self.model.training
        self.model.eval()
        tmp = tempfile.mkdtemp(prefix="ctc_infer_"); paths = []
        for i, a in enumerate(batch["_audio"]):
            p = os.path.join(tmp, f"{i}.wav")
            sf.write(p, np.asarray(a, dtype=np.float32), 16000); paths.append(p)
        hyps = self.model.transcribe(paths, batch_size=len(paths), verbose=False)
        out = [(h.text if hasattr(h, "text") else str(h)) for h in hyps]
        for p in paths:
            try: os.unlink(p)
            except Exception: pass
        try: os.rmdir(tmp)
        except Exception: pass
        if was_training: self.model.train()
        return out

## Cell 7 — Registry

In [7]:
REGISTRY: Dict[str, Callable[..., ModelAdapter]] = {
    "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0": ConformerCTCAdapter,
    "nvidia/stt_ar_fastconformer_hybrid_large_pc_v1.0":  ConformerCTCAdapter,   # same adapter, no-diacritics twin
}

def get_adapter(name, **kw) -> ModelAdapter:
    if name not in REGISTRY: raise KeyError(f"{name} not registered. Have: {list(REGISTRY)}")
    a = REGISTRY[name](name, **kw); a.name = name; return a

## Cell 8 — Datasets

Same real North-Levantine (Palestinian) Arabic corpus + soundfile decode as the sibling
notebooks. Smoke test uses clips <= 30s.

In [8]:
from datasets import load_from_disk, Audio, Dataset
import soundfile as sf, io, random as _random

REAL_DATA_DIR = Path("/workspace/asr/Palestinian-ASR/omnilingual_selected/apc_north_levantine_all_splits")

def _materialize_audio(ds):
    def gen():
        for row in ds:
            wav, sr = sf.read(io.BytesIO(row["audio"]["bytes"]), dtype="float32")
            if wav.ndim > 1: wav = wav.mean(axis=1)
            out = dict(row); out["audio"] = {"array": wav, "sampling_rate": sr}
            yield out
    return Dataset.from_generator(gen)

def load_splits(smoke=SMOKE_TEST):
    ds = load_from_disk(str(REAL_DATA_DIR))
    if "raw_text" in ds.column_names and "text" not in ds.column_names:
        ds = ds.rename_column("raw_text", "text")
    ds = ds.cast_column("audio", Audio(decode=False))
    if smoke:
        durations = ds["duration"]
        short_idx = [i for i, d in enumerate(durations) if d <= 30.0]
        _random.Random(SEED).shuffle(short_idx)
        splits = {"train": ds.select(short_idx[0:1]),
                  "validation": ds.select(short_idx[1:2]),
                  "test": ds.select(short_idx[2:3])}
    else:
        ds = ds.shuffle(seed=SEED)
        n = len(ds); n_tr, n_va = int(n * 0.8), int(n * 0.9)
        splits = {"train": ds.select(range(0, n_tr)),
                  "validation": ds.select(range(n_tr, n_va)),
                  "test": ds.select(range(n_va, n))}
    for k in splits: splits[k] = _materialize_audio(splits[k])
    return splits

SPLITS = load_splits()
{k: len(v) for k, v in SPLITS.items()}

{'train': 1, 'validation': 1, 'test': 1}

## Cell 9 — PredictAPI (cached)

In [9]:
def _fingerprint(ds) -> str:
    try: h = ds._fingerprint
    except Exception: h = str(len(ds))
    return hashlib.md5(f"{h}{len(ds)}".encode()).hexdigest()[:10]

class PredictAPI:
    @staticmethod
    def _path(model_name, split, ds, stage):
        slug = model_name.replace("/", "__")
        return PRED_DIR / f"{slug}__{split}__{_fingerprint(ds)}__{stage}.json"

    @staticmethod
    def run(adapter, ds, split="test", stage="base", batch_size=4, force=False):
        p = PredictAPI._path(adapter.name, split, ds, stage)
        if p.exists() and not force:
            print(f"[predict] CACHE HIT -> {p.name}"); return json.loads(p.read_text(encoding="utf-8"))
        print(f"[predict] generating ({stage}, {split}, n={len(ds)})")
        feats = [adapter.preprocess(ex) for ex in ds]
        preds, refs = [], []
        adapter.model.eval()
        for i in range(0, len(feats), batch_size):
            b = adapter.collate(feats[i:i+batch_size])
            preds.extend(adapter.generate(b)); refs.extend(b["text"])
            print(f"  {min(i+batch_size,len(feats))}/{len(feats)}", end="\r")
        rec = {"model": adapter.name, "split": split, "stage": stage,
               "predictions": preds, "references": refs, "n": len(preds), "ts": time.time()}
        p.write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"\n[predict] saved -> {p.name}")
        return rec

## Cell 10 — EvaluateAPI (cached)

In [10]:
class EvaluateAPI:
    @staticmethod
    def _path(model_name, split, stage, pred_record=None):
        slug = model_name.replace('/', '__')
        if pred_record is None:
            return METRIC_DIR / f"{slug}__{split}__{stage}.json"
        # Content-address the metric to the exact predictions it scores. PredictAPI already
        # keys on the dataset fingerprint; without the same discipline here a metric computed
        # on an older corpus gets silently re-served after the data changes, producing a wrong
        # base WER and a meaningless base->tuned delta.
        payload = json.dumps([pred_record["predictions"], pred_record["references"]],
                             ensure_ascii=False, sort_keys=True).encode("utf-8")
        h = hashlib.md5(payload).hexdigest()[:10]
        return METRIC_DIR / f"{slug}__{split}__{h}__{stage}.json"

    @staticmethod
    def run(model_name, pred_record, split="test", stage="base", force=False):
        p = EvaluateAPI._path(model_name, split, stage, pred_record)
        if p.exists() and not force:
            m = json.loads(p.read_text()); print(f"[eval] CACHE HIT -> {m}"); return m
        m = compute_wer_cer(pred_record["predictions"], pred_record["references"])
        m.update({"model": model_name, "split": split, "stage": stage})
        p.write_text(json.dumps(m, indent=2))
        print(f"[eval] WER={m['wer']:.4f} CER={m['cer']:.4f} (n={m['n']}) -> {p.name}")
        return m

## Cell 11 — Build adapter + load base model

In [11]:
set_seed()
adapter = get_adapter(MODEL_NAME, lang=LANG)
adapter.load_base()
n_params = sum(p.numel() for p in adapter.model.parameters())
print(f"{MODEL_NAME}: {n_params/1e6:.1f}M params | loss_type={adapter.loss_type}")

[NeMo W 2026-07-31 19:47:55 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.


OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.


No exporters were provided. This means that no telemetry data will be collected.


[load] nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0


[NeMo W 2026-07-31 19:48:18 model_utils:515] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': True, 'num_workers': 8, 'pin_memory': True, 'max_duration': 20, 'min_duration': 0.5, 'is_tarred': True, 'tarred_audio_filepaths': '???', 'shuffle_n': 2048, 'bucketing_strategy': 'fully_randomized', 'bucketing_batch_size': None}
     Reason: Missing mandatory value: train_ds.manifest_filepath
        full_key: train_ds.manifest_filepath
        object_type=dict.


...

[NeMo I 2026-07-31 19:48:21 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo I 2026-07-31 19:48:22 save_restore_connector:285] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /workspace/asr_env/models/hf/hub/models--nvidia--stt_ar_fastconformer_hybrid_large_pcd_v1.0/snapshots/7f32349d952f42a28dce979ba73270aa2bbdfa89/stt_ar_fastconformer_hybrid_large_pcd_v1.0.nemo.


[NeMo I 2026-07-31 19:48:22 hybrid_rnnt_ctc_bpe_models:488] No `decoding_cfg` passed when changing decoding strategy, using internal config


[NeMo I 2026-07-31 19:48:22 hybrid_rnnt_ctc_bpe_models:513] Changed decoding strategy of the CTC decoder to 
    strategy: greedy
    preserve_alignments: null
    compute_timestamps: null
    word_seperator: ' '
    segment_seperators:
    - .
    - '!'
    - '?'
    segment_gap_threshold: null
    ctc_timestamp_type: all
    batch_dim_index: 0
    greedy:
      preserve_alignments: false
      compute_timestamps: false
      preserve_frame_confidence: false
      confidence_method_cfg:
        name: entropy
        entropy_type: tsallis
        alpha: 0.33
        entropy_norm: exp
        temperature: DEPRECATED
      ngram_lm_model: null
      ngram_lm_alpha: 0.0
      boosting_tree:
        model_path: null
        key_phrases_file: null
        key_phrases_list: null
        key_phrase_items_list: null
        context_score: 1.0
        depth_scaling: 2.0
        unk_score: 0.0
        final_eos_score: 1.0
        score_per_phrase: 0.0
        source_lang: en
        use_triton: 

nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0: 114.6M params | loss_type=ctc


## Cell 12 — Baseline preds + eval on test (cached)

In [12]:
base_preds   = PredictAPI.run(adapter, SPLITS["test"], split="test", stage="base",
                              batch_size=ConfigAPI.train(MODEL_NAME).per_device_eval_batch_size)
base_metrics = EvaluateAPI.run(MODEL_NAME, base_preds, split="test", stage="base")
for p, r in list(zip(base_preds["predictions"], base_preds["references"]))[:3]:
    print(f"REF : {r}\nHYP : {p}\n")

[predict] CACHE HIT -> nvidia__stt_ar_fastconformer_hybrid_large_pcd_v1.0__test__15bf14064d__base.json
[eval] CACHE HIT -> {'wer': 0.4146341463414634, 'cer': 0.16591928251121077, 'n': 1, 'model': 'nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0', 'split': 'test', 'stage': 'base'}
REF : بعدين منحط البهارات فوين وبعد ما منحط البهارات فوين منخلين يغلو ليستو شوي بعدين منشلح فوين الفريكه وبس تستوي مناكلا ولا اطيب من هيك بتاخد مده الاستوا noise لالا تاريبا حوالي شي ساعه علي الغاز مجرد ما تستوي بتصير جاهزه للاكل
HYP : بعدين بنحط البهارات فوقهم. وبعد ما بنحط البهارات فوقهم بنخليهم يغلووا ليسستووا شوي بعدين بنشلح فوقهم الفريكة. وبس بتستوي ناأكللها ولا أطيب من هيك. بتاخد مدة الاستوا لها تقريبًا حوالي شي ساعة على الغااز مجرد ما تستوي، بصير جاههزة للأكل.



## Cell 13 — Apply LoRA

In [13]:
lora_spec  = ConfigAPI.lora(MODEL_NAME)
train_spec = ConfigAPI.train(MODEL_NAME)
if SMOKE_TEST:
    train_spec.num_epochs = 2
    train_spec.early_stopping_patience = 4
adapter.apply_lora(lora_spec)

trainable params: 7,798,784 || all params: 122,420,226 || trainable%: 6.3705
[lora] peft


PeftModel(
  (base_model): LoraModel(
    (model): EncDecHybridRNNTCTCBPEModel(
      (preprocessor): AudioToMelSpectrogramPreprocessor(
        (featurizer): FilterbankFeatures()
      )
      (encoder): ConformerEncoder(
        (pre_encode): ConvSubsampling(
          (out): Linear(in_features=2560, out_features=512, bias=True)
          (conv): MaskedConvSequential(
            (0): Conv2d(1, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
            (1): ReLU(inplace=True)
            (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
            (3): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
            (4): ReLU(inplace=True)
            (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
            (6): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
            (7): ReLU(inplace=True)
          )
        )
        (pos_enc): RelPositionalEncoding(
          (dropout): Dropout(p=0.1, inpl

## Cell 14 — TrainAPI

Same custom loop as the sibling notebooks (per-epoch train loss / val loss / val WER / val CER,
early stopping on WER patience 4, best-WER checkpoint). CTC-specific points: `train_step`
computes the CTC loss on `self.model`, the optimizer trains `adapter.trainable_parameters()`
(the injected LoRA tensors), and the best checkpoint is a real PEFT adapter saved via
`adapter.save_pretrained`.

In [14]:
import mlflow
from contextlib import nullcontext
from torch.utils.data import DataLoader, Sampler

def _amp(spec):
    """bf16 autocast on CUDA; no-op elsewhere so the loop also runs on CPU."""
    if DEVICE == "cuda":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    return nullcontext()

class _ListDS(torch.utils.data.Dataset):
    def __init__(self, feats): self.f = feats
    def __len__(self): return len(self.f)
    def __getitem__(self, i): return self.f[i]

class LengthGroupedSampler(Sampler):
    """Buckets examples into batch_size-sized windows sorted by audio length (for padding
    efficiency), then shuffles the *order of windows* every epoch -- so training still sees
    a different batch order each pass instead of a frozen short->long sweep. A fresh
    torch.randperm before the sort also jitters which items land in which window across
    epochs (tie-breaking), not just the window order."""
    def __init__(self, lengths, batch_size):
        self.lengths = lengths; self.batch_size = batch_size
    def __len__(self): return len(self.lengths)
    def __iter__(self):
        idx = torch.randperm(len(self.lengths)).tolist()
        idx.sort(key=lambda i: self.lengths[i])
        buckets = [idx[i:i + self.batch_size] for i in range(0, len(idx), self.batch_size)]
        order = torch.randperm(len(buckets)).tolist()
        out = []
        for b in order: out.extend(buckets[b])
        return iter(out)

def _rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def _restore_rng(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])

class TrainAPI:
    @staticmethod
    def _prep(adapter, ds, spec):
        """Pre-flight: preprocess + drop over-long / empty items before the loop."""
        feats, dropped = [], 0
        for ex in ds:
            f = adapter.preprocess(ex)
            if f["audio_len"] > spec.max_audio_seconds: dropped += 1; continue
            if not f["text"].strip():                   dropped += 1; continue
            if len(f.get("labels") or []) > spec.max_label_tokens:
                f["labels"] = f["labels"][:spec.max_label_tokens]
            feats.append(f)
        print(f"[prep] kept {len(feats)}, dropped {dropped}")
        return feats

    @staticmethod
    @torch.no_grad()
    def _validate(adapter, loader, spec):
        adapter.model.eval(); losses, preds, refs = [], [], []
        for b in loader:
            g = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
            try:
                with _amp(spec):
                    losses.append(float(adapter.train_step(g)))
            except Exception as e:
                print(f"[val] loss skipped: {e}")
            preds.extend(adapter.generate(g)); refs.extend(b["text"])
        m = compute_wer_cer(preds, refs)
        m["val_loss"] = float(np.mean(losses)) if losses else float("nan")
        m["_preds"], m["_refs"] = preds, refs
        return m

    @staticmethod
    def run(adapter, splits, spec: TrainConfigSpec, lora_spec: LoRAConfigSpec, resume: bool = True):
        slug = adapter.name.replace("/", "__")
        run_root = CKPT_DIR / slug
        best_dir = run_root / "best"; best_dir.mkdir(parents=True, exist_ok=True)

        tr_f = TrainAPI._prep(adapter, splits["train"], spec)
        va_f = TrainAPI._prep(adapter, splits["validation"], spec)
        tr_kw = {}
        if spec.dataloader_num_workers > 0:
            tr_kw["persistent_workers"] = spec.dataloader_persistent_workers
            tr_kw["prefetch_factor"] = spec.dataloader_prefetch_factor
        tr = DataLoader(_ListDS(tr_f), batch_size=spec.per_device_train_batch_size,
                        sampler=LengthGroupedSampler([f["audio_len"] for f in tr_f],
                                                     spec.per_device_train_batch_size),
                        collate_fn=adapter.collate, num_workers=spec.dataloader_num_workers,
                        pin_memory=(spec.dataloader_pin_memory and DEVICE == "cuda"),
                        drop_last=False, **tr_kw)
        va = DataLoader(_ListDS(va_f), batch_size=spec.per_device_eval_batch_size, shuffle=False,
                        collate_fn=adapter.collate, num_workers=spec.dataloader_num_workers)

        params = adapter.trainable_parameters()
        opt = None
        if DEVICE == "cuda":            # bitsandbytes 8-bit optimizers are CUDA-only
            try:
                import bitsandbytes as bnb
                opt = bnb.optim.AdamW8bit(params, lr=spec.learning_rate, weight_decay=spec.weight_decay)
            except Exception as e:
                print(f"[opt] AdamW8bit unavailable ({e}); using torch.AdamW")
        if opt is None:
            opt = torch.optim.AdamW(params, lr=spec.learning_rate, weight_decay=spec.weight_decay)

        steps_pe = max(1, math.ceil(len(tr) / spec.gradient_accumulation_steps))
        total    = steps_pe * spec.num_epochs
        save_steps = spec.save_steps or max(200, steps_pe // 3)
        from transformers import get_linear_schedule_with_warmup
        sched = get_linear_schedule_with_warmup(opt, int(total * spec.warmup_ratio), total)

        best_wer, bad_epochs, gstep, history, start_epoch, mlflow_run_id = \
            float("inf"), 0, 0, [], 1, None

        ckpts = sorted(run_root.glob("ckpt_step*"))
        if resume and ckpts:
            last = ckpts[-1]
            try:
                state = json.loads((last / "trainer_state.json").read_text())
                adapter.load_checkpoint(last)
                # weights_only=False: these are our own trusted local checkpoint files, not
                # untrusted downloads. Needed because torch >=2.6 defaults weights_only=True,
                # which rejects the numpy-backed RNG state (numpy.random.get_state() pickles via
                # numpy's own _reconstruct, not in the default safe-globals allowlist) and can
                # also reject optimizer state depending on the optimizer's internals.
                opt.load_state_dict(torch.load(last / "optimizer.pt", map_location=DEVICE, weights_only=False))
                sched.load_state_dict(torch.load(last / "scheduler.pt", map_location=DEVICE, weights_only=False))
                _restore_rng(torch.load(last / "rng.pt", map_location="cpu", weights_only=False))
                best_wer, bad_epochs = state["best_wer"], state["bad_epochs"]
                gstep, start_epoch = state["gstep"], state["epoch"] + 1
                history, mlflow_run_id = state["history"], state.get("mlflow_run_id")
                print(f"[resume] epoch {start_epoch} gstep {gstep} best_wer {best_wer:.4f} <- {last}")
            except Exception as e:
                print(f"[resume] failed ({e}); starting fresh")

        try:
            mlflow.start_run(run_id=mlflow_run_id, run_name=f"{slug}-lora")
        except Exception as e:
            # A resumed run_id can be unusable for reasons outside our control (belongs to
            # a different active experiment -- e.g. this same checkpoint dir was previously
            # used by a sibling notebook under a different MLflow experiment name; a
            # manually deleted run; a different MLFLOW_TRACKING_URI). Never let bookkeeping
            # block real training -- fall back to a fresh run instead of crashing.
            print(f"[mlflow] could not resume run {mlflow_run_id} ({e}); starting a new run")
            mlflow_run_id = None
            mlflow.start_run(run_id=None, run_name=f"{slug}-lora")
        mlflow_run_id = mlflow.active_run().info.run_id
        try:
            params_flat = {f"train.{k}": (str(v) if isinstance(v, (list, type(None))) else v)
                           for k, v in asdict(spec).items()}
            params_flat.update({f"lora.{k}": (str(v) if isinstance(v, (list, type(None))) else v)
                                for k, v in asdict(lora_spec).items()})
            params_flat.update({"model": adapter.name, "lang": LANG, "smoke": SMOKE_TEST,
                                "save_steps": save_steps})
            mlflow.log_params(params_flat)
        except Exception as e:
            print(f"[mlflow] log_params skipped ({e})")
        try:
            mlflow.set_tags({"dataset_version": REAL_DATA_DIR.name,
                             "hardware": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
                             "stage": "dev" if SMOKE_TEST else "experiment"})
        except Exception as e:
            print(f"[mlflow] set_tags skipped ({e})")

        def _save_numbered_ckpt(epoch):
            nonlocal gstep
            d = run_root / f"ckpt_step{gstep:08d}"; d.mkdir(parents=True, exist_ok=True)
            adapter.save_checkpoint(d)
            torch.save(opt.state_dict(), d / "optimizer.pt")
            torch.save(sched.state_dict(), d / "scheduler.pt")
            torch.save(_rng_state(), d / "rng.pt")
            (d / "trainer_state.json").write_text(json.dumps(
                {"epoch": epoch, "gstep": gstep, "best_wer": best_wer, "bad_epochs": bad_epochs,
                 "history": history, "mlflow_run_id": mlflow_run_id}, indent=2))
            kept = sorted(run_root.glob("ckpt_step*"))
            for old in kept[:-spec.save_total_limit] if spec.save_total_limit > 0 else kept:
                shutil.rmtree(old, ignore_errors=True)
            return d

        for epoch in range(start_epoch, spec.num_epochs + 1):
            t_epoch = time.time()
            adapter.model.train(); ep_loss, nb = 0.0, 0
            opt.zero_grad(set_to_none=True)
            for i, b in enumerate(tr):
                t_step = time.time()
                g = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
                with _amp(spec):
                    loss = adapter.train_step(g) / spec.gradient_accumulation_steps
                if not torch.isfinite(loss):
                    print(f"[nan] step {i} non-finite loss, skipping batch")
                    opt.zero_grad(set_to_none=True); continue
                loss.backward()
                if (i + 1) % spec.gradient_accumulation_steps == 0 or (i + 1) == len(tr):
                    gnorm = torch.nn.utils.clip_grad_norm_(params, spec.max_grad_norm)
                    if not torch.isfinite(gnorm):
                        print(f"[nan] step {i} non-finite grad norm, skipping update")
                        opt.zero_grad(set_to_none=True); continue
                    opt.step(); sched.step(); opt.zero_grad(set_to_none=True); gstep += 1
                    mem = torch.cuda.memory_allocated() / 1e9 if DEVICE == "cuda" else 0.0
                    mlflow.log_metrics({"train/step_loss": float(loss) * spec.gradient_accumulation_steps,
                                        "train/grad_norm": float(gnorm),
                                        "train/lr": sched.get_last_lr()[0],
                                        "train/step_time_s": time.time() - t_step,
                                        "train/gpu_mem_gb": mem}, step=gstep)
                    if gstep % save_steps == 0:
                        _save_numbered_ckpt(epoch)
                ep_loss += float(loss) * spec.gradient_accumulation_steps; nb += 1

            train_loss = ep_loss / max(nb, 1)
            vm = TrainAPI._validate(adapter, va, spec)
            row = {"epoch": epoch, "train_loss": train_loss, "val_loss": vm["val_loss"],
                   "val_wer": vm["wer"], "val_cer": vm["cer"]}
            history.append(row)
            throughput = sum(f["audio_len"] for f in tr_f) / max(time.time() - t_epoch, 1e-6)
            mlflow.log_metrics({"epoch": epoch, "train/loss": train_loss, "val/loss": vm["val_loss"],
                                "val/wer": vm["wer"], "val/cer": vm["cer"],
                                "train/throughput_audio_s_per_s": throughput,
                                "train/epoch_time_s": time.time() - t_epoch}, step=gstep)
            try:
                import pandas as pd
                qual = pd.DataFrame({"reference": vm["_refs"][:5], "hypothesis": vm["_preds"][:5]})
                mlflow.log_table(qual, artifact_file=f"qualitative/epoch_{epoch:03d}.json")
            except Exception as e:
                print(f"[mlflow] qualitative table skipped ({e})")
            print(f"epoch {epoch:>3} | train {train_loss:.4f} | val {vm['val_loss']:.4f} "
                  f"| WER {vm['wer']:.4f} | CER {vm['cer']:.4f}")

            # ---- best-WER checkpoint (protected) + early stopping ----
            if vm["wer"] < best_wer - 1e-6:
                best_wer, bad_epochs = vm["wer"], 0
                adapter.save_checkpoint(best_dir)
                (best_dir / "best.json").write_text(json.dumps({**row, "gstep": gstep}, indent=2))
                print(f"  -> new best WER {best_wer:.4f}, saved to {best_dir}")
            else:
                bad_epochs += 1
                print(f"  -> no improvement ({bad_epochs}/{spec.early_stopping_patience})")

            # ---- epoch-boundary checkpoint, regardless of step count: clean resume fallback ----
            _save_numbered_ckpt(epoch)

            if bad_epochs >= spec.early_stopping_patience:
                print(f"[early-stop] epoch {epoch}, best WER {best_wer:.4f}"); break

        mlflow.log_metric("best_val_wer", best_wer, step=gstep)
        (run_root / "history.json").write_text(json.dumps(history, indent=2))
        return {"best_wer": best_wer, "best_dir": str(best_dir), "history": history,
                "mlflow_run_id": mlflow_run_id}


## Cell 15 — Train

In [15]:
set_seed()
train_out = TrainAPI.run(adapter, SPLITS, train_spec, lora_spec)
print(f"best val WER: {train_out['best_wer']:.4f} @ {train_out['best_dir']}")

[prep] kept 1, dropped 0


[prep] kept 1, dropped 0


[NeMo W 2026-07-31 19:49:28 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token


[NeMo W 2026-07-31 19:49:28 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


[NeMo W 2026-07-31 19:49:28 ctc_greedy_decoding:277] CTC decoding strategy 'greedy' is slower than 'greedy_batch', which implements the same exact interface. Consider changing your strategy to 'greedy_batch' for a free performance improvement.


...

  -> new best WER 0.5581, saved to /workspace/asr_env/checkpoints/nvidia__stt_ar_fastconformer_hybrid_large_pcd_v1.0/best


[NeMo W 2026-07-31 19:49:30 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token


[NeMo W 2026-07-31 19:49:30 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


epoch   2 | train 994.2365 | val 423.7295 | WER 0.5581 | CER 0.2457
  -> no improvement (1/4)


best val WER: 0.5581 @ /workspace/asr_env/checkpoints/nvidia__stt_ar_fastconformer_hybrid_large_pcd_v1.0/best


## Cell 16 — Load best checkpoint, predict + evaluate on test, save

In [16]:
best = Path(train_out["best_dir"])
try:
    adapter.load_checkpoint(best)
    print(f"[ckpt] restored best adapter <- {best}")
except Exception as e:
    print(f"[ckpt] restore failed ({e})")

tuned_preds   = PredictAPI.run(adapter, SPLITS["test"], split="test", stage="tuned",
                               batch_size=train_spec.per_device_eval_batch_size, force=True)
tuned_metrics = EvaluateAPI.run(MODEL_NAME, tuned_preds, split="test", stage="tuned", force=True)

dataset_hours  = {k: sum(v["duration"]) / 3600.0 for k, v in SPLITS.items()}
dataset_counts = {k: len(v) for k, v in SPLITS.items()}

summary = {
    "model": MODEL_NAME, "lang": LANG, "smoke_test": SMOKE_TEST,
    "base":  {"wer": base_metrics["wer"],  "cer": base_metrics["cer"]},
    "tuned": {"wer": tuned_metrics["wer"], "cer": tuned_metrics["cer"]},
    "delta": {"wer": base_metrics["wer"] - tuned_metrics["wer"],
              "cer": base_metrics["cer"] - tuned_metrics["cer"]},
    "best_val_wer": train_out["best_wer"],
    "dataset_counts": dataset_counts, "dataset_hours": dataset_hours,
    "lora": asdict(lora_spec), "train": asdict(train_spec),
}
sp = METRIC_DIR / f"{MODEL_NAME.replace('/','__')}__SUMMARY.json"
sp.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

mlflow.log_metrics({"test/base_wer": base_metrics["wer"],   "test/base_cer": base_metrics["cer"],
                    "test/tuned_wer": tuned_metrics["wer"], "test/tuned_cer": tuned_metrics["cer"],
                    **{f"data/{k}_hours": v for k, v in dataset_hours.items()},
                    **{f"data/{k}_count": v for k, v in dataset_counts.items()}})
try:
    mlflow.log_artifact(str(sp), artifact_path="summary")
    import pandas as pd
    mlflow.log_table(pd.DataFrame({"reference": tuned_preds["references"],
                                   "base_hyp": base_preds["predictions"],
                                   "tuned_hyp": tuned_preds["predictions"]}),
                     artifact_file="qualitative/test_predictions.json")
except Exception as e:
    print(f"[mlflow] artifact logging skipped ({e})")
mlflow.end_run()
print(json.dumps(summary, indent=2, ensure_ascii=False))


[ckpt] restored best adapter <- /workspace/asr_env/checkpoints/nvidia__stt_ar_fastconformer_hybrid_large_pcd_v1.0/best
[predict] generating (tuned, test, n=1)


[NeMo W 2026-07-31 19:49:32 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token


[NeMo W 2026-07-31 19:49:32 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


  1/1
[predict] saved -> nvidia__stt_ar_fastconformer_hybrid_large_pcd_v1.0__test__15bf14064d__tuned.json
[eval] WER=0.3902 CER=0.1435 (n=1) -> nvidia__stt_ar_fastconformer_hybrid_large_pcd_v1.0__test__e58acb1a11__tuned.json


{
  "model": "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0",
  "lang": "ar",
  "smoke_test": true,
  "base": {
    "wer": 0.4146341463414634,
    "cer": 0.16591928251121077
  },
  "tuned": {
    "wer": 0.3902439024390244,
    "cer": 0.14349775784753363
  },
  "delta": {
    "wer": 0.02439024390243899,
    "cer": 0.02242152466367714
  },
  "best_val_wer": 0.5581395348837209,
  "dataset_counts": {
    "train": 1,
    "validation": 1,
    "test": 1
  },
  "dataset_hours": {
    "train": 0.008036799768518519,
    "validation": 0.0077747627314814815,
    "test": 0.006103252314814815
  },
  "lora": {
    "r": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "bias": "none",
    "target_modules": [
      "linear_q",
      "linear_k",
      "linear_v",
      "linear_out",
      "linear1",
      "linear2"
    ],
    "modules_to_save": null,
    "task_type": null
  },
  "train": {
    "num_epochs": 2,
    "early_stopping_patience": 4,
    "metric_for_best": "wer",
    "greater_is_bett

## Cell 17 — One-shot smoke wrapper

Same code path end-to-end (load → base predict/eval → LoRA → train → tuned predict/eval),
callable for any registered FastConformer-CTC checkpoint.

In [17]:
def smoke(model_name, splits):
    set_seed()
    a = get_adapter(model_name, lang=LANG); a.load_base()
    bp = PredictAPI.run(a, splits["test"], "test", "base", batch_size=2)
    bm = EvaluateAPI.run(model_name, bp, "test", "base")
    ls, ts = ConfigAPI.lora(model_name), ConfigAPI.train(model_name)
    ts.num_epochs = 2; ts.per_device_train_batch_size = 1; ts.gradient_accumulation_steps = 2
    a.apply_lora(ls)
    out = TrainAPI.run(a, splits, ts, ls)
    tp = PredictAPI.run(a, splits["test"], "test", "tuned", batch_size=2, force=True)
    tm = EvaluateAPI.run(model_name, tp, "test", "tuned", force=True)
    if mlflow.active_run() is not None:   # TrainAPI.run() started one; this loop never reaches
        mlflow.end_run()                  # the Cell 16 save-results cell that normally ends it
    del a.model, a; gc.collect(); torch.cuda.empty_cache()
    return {"model": model_name, "base_wer": bm["wer"], "tuned_wer": tm["wer"],
            "best_val_wer": out["best_wer"], "status": "PASS"}

results = []
for m in ["nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0"]:
    try:
        results.append(smoke(m, SPLITS))
    except Exception as e:
        import traceback; traceback.print_exc()
        results.append({"model": m, "status": f"FAIL: {e}"})
    print("=" * 70)

import pandas as pd
pd.DataFrame(results)

[load] nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0


[NeMo W 2026-07-31 19:49:41 model_utils:515] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': True, 'num_workers': 8, 'pin_memory': True, 'max_duration': 20, 'min_duration': 0.5, 'is_tarred': True, 'tarred_audio_filepaths': '???', 'shuffle_n': 2048, 'bucketing_strategy': 'fully_randomized', 'bucketing_batch_size': None}
     Reason: Missing mandatory value: train_ds.manifest_filepath
        full_key: train_ds.manifest_filepath
        object_type=dict.


[NeMo W 2026-07-31 19:49:41 model_utils:515] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': False, 'use_start_end_token': False, 'num_workers': 8, 'pin_memory': True}
     Reason: Missing mandatory value: validation_ds.manifest_filepath
        full_key: validation_ds.manifest_filepath
        object_type=dict.


[NeMo W 2026-07-31 19:49:41 model_utils:515] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': False, 'use_start_end_token': False, 'num_workers': 8, 'pin_memory': True}
     Reason: Missing mandatory value: test_ds.manifest_filepath
        full_key: test_ds.manifest_filepath
        object_type=dict.


[NeMo W 2026-07-31 19:49:41 model_utils:515] Skipped conversion for config/subconfig:
    {'dir': '???', 'type': 'bpe', 'model_path': 'nemo:43c84e71237048ddab3bb273bdc00fa0_tokenizer.model', 'vocab_path': 'nemo:1e7bbe36b91a472896c99283bda08bf3_vocab.txt', 'spe_tokenizer_vocab': 'nemo:e7a019581cd54cce88ac2acf8e4c54a3_tokenizer.vocab'}
     Reason: Missing mandatory value: tokenizer.dir
        full_key: tokenizer.dir
        object_type=dict.


...

[predict] generating (tuned, test, n=1)


[NeMo W 2026-07-31 19:49:49 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token


[NeMo W 2026-07-31 19:49:49 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


  1/1
[predict] saved -> nvidia__stt_ar_fastconformer_hybrid_large_pcd_v1.0__test__15bf14064d__tuned.json
[eval] WER=0.4146 CER=0.1659 (n=1) -> nvidia__stt_ar_fastconformer_hybrid_large_pcd_v1.0__test__0d4f747c4a__tuned.json


,model,base_wer,tuned_wer,best_val_wer,status
0,nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0,0.414634,0.414634,0.55814,PASS
